| Python | 3.10 |

| mediapipe | 0.10.9 |

| opencv-python | 4.8.0.76 |

| protobuf | 4.25.3 |

| numpy | 1.24.3 |


import 

import mediapipe as mp

print("cv2:", cv2.__version__)

print("mediapipe:", mp.__version__)

print("Python:", sys.version)

### 캠버전

In [4]:


import cv2
import mediapipe as mp
import math

mp_drawing = mp.solutions.drawing_utils
mp_drawing_styles = mp.solutions.drawing_styles
mp_hands = mp.solutions.hands

cap = cv2.VideoCapture(0)

# ─── 설정 ───────────────────────────────────────
CLICK_DISTANCE = 40   # 엄지-검지 거리 이하면 클릭
PANEL_HEIGHT   = 160   # 상단 UI 패널 높이

# 현재 선택된 값
current_color     = (0, 255, 0)   # 초록
current_thickness = 5

index_trail = []
all_trails  = []   # (좌표리스트, 색상, 굵기) 묶어서 저장

# ─── UI 버튼 정의 ────────────────────────────────
# (라벨, BGR색상, 버튼색상)
color_buttons = [
    ("RED",   (0, 0, 255),   (0, 0, 255)),
    ("GREEN", (0, 255, 0),   (0, 255, 0)),
    ("BLUE",  (255, 0, 0),   (255, 0, 0)),
    ("WHITE", (255,255,255), (200,200,200)),
    ("YELLOW",(0, 255, 255), (0, 255, 255)),
]

thickness_buttons = [
    ("THIN",   3),
    ("NORMAL", 7),
    ("THICK",  13),
]
# ─────────────────────────────────────────────────

def draw_panel(frame, w):
    cv2.rectangle(frame, (0, 0), (w, PANEL_HEIGHT), (40, 40, 40), -1)

    btn_w = 70
    buttons = []

    # 색상 버튼 (1행)
    for i, (label, color, btn_color) in enumerate(color_buttons):
        x1 = 10 + i * (btn_w + 8)
        y1, x2, y2 = 10, x1 + btn_w, 70  # y1=10

        border = 3 if current_color == color else 1
        cv2.rectangle(frame, (x1, y1), (x2, y2), btn_color, -1)
        cv2.rectangle(frame, (x1, y1), (x2, y2), (255,255,255), border)
        cv2.putText(frame, label, (x1+5, y1+38),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.45, (0,0,0), 2)
        buttons.append((x1, y1, x2, y2, 'color', color))

    # 굵기 버튼 (2행 - 아래로 내림)
    for i, (label, thick) in enumerate(thickness_buttons):
        x1 = 10 + i * (btn_w + 8)
        y1, x2, y2 = 90, x1 + btn_w, 150  # y1=90으로 내림

        border = 3 if current_thickness == thick else 1
        cv2.rectangle(frame, (x1, y1), (x2, y2), (80, 80, 80), -1)
        cv2.rectangle(frame, (x1, y1), (x2, y2), (255,255,255), border)
        cv2.putText(frame, label, (x1+5, y1+38),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.45, (255,255,255), 2)
        buttons.append((x1, y1, x2, y2, 'thickness', thick))

    # CLEAR 버튼 (2행 굵기버튼 옆)
    cx1 = 10 + len(thickness_buttons) * (btn_w + 8) + 10
    cy1, cx2, cy2 = 90, cx1 + btn_w, 150  # y1=90으로 내림
    cv2.rectangle(frame, (cx1, cy1), (cx2, cy2), (0, 0, 180), -1)
    cv2.rectangle(frame, (cx1, cy1), (cx2, cy2), (255,255,255), 1)
    cv2.putText(frame, "CLEAR", (cx1+5, cy1+38),
                cv2.FONT_HERSHEY_SIMPLEX, 0.45, (255,255,255), 2)
    buttons.append((cx1, cy1, cx2, cy2, 'clear', None))

    return buttons


def check_button_click(ix, iy, buttons):
    """검지 좌표가 버튼 영역 안에 있는지 확인"""
    for (x1, y1, x2, y2, btn_type, value) in buttons:
        if x1 < ix < x2 and y1 < iy < y2:
            return btn_type, value
    return None, None


with mp_hands.Hands(
    static_image_mode=False,
    max_num_hands=1,
    model_complexity=1,
    min_detection_confidence=0.5,
    min_tracking_confidence=0.5
) as hands:

    while cap.isOpened():
        success, frame = cap.read()
        if not success:
            continue

        frame = cv2.flip(frame, 1)
        h, w, _ = frame.shape

        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        frame_rgb.flags.writeable = False
        results = hands.process(frame_rgb)
        frame_rgb.flags.writeable = True
        frame = cv2.cvtColor(frame_rgb, cv2.COLOR_RGB2BGR)

        # UI 패널 그리기 + 버튼 목록 받기
        buttons = draw_panel(frame, w)

        if results.multi_hand_landmarks:
            for hand_landmarks in results.multi_hand_landmarks:


                # 검지 끝 (8번)
                index_tip = hand_landmarks.landmark[8]
                ix = int(index_tip.x * w)
                iy = int(index_tip.y * h)

                # 엄지 끝 (4번)
                thumb_tip = hand_landmarks.landmark[4]
                tx = int(thumb_tip.x * w)
                ty = int(thumb_tip.y * h)

                distance = math.sqrt((ix-tx)**2 + (iy-ty)**2)

                # 검지 커서 표시
                cv2.circle(frame, (ix, iy), 8, current_color, -1)

                if distance < CLICK_DISTANCE:
                    # ── 클릭 상태 ──
                    cv2.circle(frame, (ix, iy), 15, (0, 0, 255), 2)

                    # 패널 영역 클릭인지 확인
                    btn_type, value = check_button_click(ix, iy, buttons)

                    if btn_type == 'color':
                        current_color = value
                        if index_trail:
                            all_trails.append((index_trail.copy(), current_color, current_thickness))
                            index_trail.clear()

                    elif btn_type == 'thickness':
                        current_thickness = value
                        if index_trail:
                            all_trails.append((index_trail.copy(), current_color, current_thickness))
                            index_trail.clear()

                    elif btn_type == 'clear':
                        index_trail.clear()
                        all_trails.clear()

                    else:
                        # 패널 밖 클릭 → 펜 들기
                        if index_trail:
                            all_trails.append((index_trail.copy(), current_color, current_thickness))
                            index_trail.clear()

                else:
                    # ── 그리기 상태 (패널 아래 영역만) ──
                    if iy > PANEL_HEIGHT:
                        index_trail.append((ix, iy))

        else:
            if index_trail:
                all_trails.append((index_trail.copy(), current_color, current_thickness))
                index_trail.clear()

        # 저장된 선 전부 그리기
        for (trail, color, thickness) in all_trails:
            for i in range(1, len(trail)):
                cv2.line(frame, trail[i-1], trail[i], color, thickness)

        # 현재 그리는 선
        for i in range(1, len(index_trail)):
            cv2.line(frame, index_trail[i-1], index_trail[i], current_color, current_thickness)

        # 현재 설정 표시
        cv2.putText(frame, f"Q:종료  C:전체지우기",
                    (10, h-15), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (200,200,200), 1)

        cv2.imshow("전자칠판", frame)

        key = cv2.waitKey(5)
        if key == ord('q'):
            break
        elif key == ord('c'):
            index_trail.clear()
            all_trails.clear()

cap.release()
cv2.destroyAllWindows()

### 동영상 버전

In [20]:
import cv2
import mediapipe as mp
import math

mp_drawing = mp.solutions.drawing_utils
mp_drawing_styles = mp.solutions.drawing_styles
mp_hands = mp.solutions.hands

cap = cv2.VideoCapture("hands3.mp4")

# ─── 설정 ───────────────────────────────────────
CLICK_DISTANCE = 40   # 엄지-검지 거리 이하면 클릭
PANEL_HEIGHT   = 160   # 상단 UI 패널 높이

# 현재 선택된 값
current_color     = (0, 255, 0)   # 초록
current_thickness = 5

index_trail = []
all_trails  = []   # (좌표리스트, 색상, 굵기) 묶어서 저장

# ─── UI 버튼 정의 ────────────────────────────────
# (라벨, BGR색상, 버튼색상)
color_buttons = [
    ("R",   (0, 0, 255),   (0, 0, 255)),
    ("G", (0, 255, 0),   (0, 255, 0)),
    ("B",  (255, 0, 0),   (255, 0, 0)),
    ("W", (255,255,255), (200,200,200)),
    ("Y",(0, 255, 255), (0, 255, 255)),
]

thickness_buttons = [
    ("THIN",   1),
    ("NORMAL", 5),
    ("THICK",  10),
]
# ─────────────────────────────────────────────────

def draw_panel(frame, w):
    cv2.rectangle(frame, (0, 0), (w, PANEL_HEIGHT), (40, 40, 40), -1)

    btn_w = 70
    buttons = []

    # 색상 버튼 (1행)
    for i, (label, color, btn_color) in enumerate(color_buttons):
        x1 = 10 + i * (btn_w + 8)
        y1, x2, y2 = 10, x1 + btn_w, 70  # y1=10

        border = 3 if current_color == color else 1
        cv2.rectangle(frame, (x1, y1), (x2, y2), btn_color, -1)
        cv2.rectangle(frame, (x1, y1), (x2, y2), (255,255,255), border)
        buttons.append((x1, y1, x2, y2, 'color', color))

    # 굵기 버튼 (2행 - 아래로 내림)
    for i, (label, thick) in enumerate(thickness_buttons):
        x1 = 10 + i * (btn_w + 8)
        y1, x2, y2 = 90, x1 + btn_w, 150  # y1=90으로 내림

        border = 3 if current_thickness == thick else 1
        cv2.rectangle(frame, (x1, y1), (x2, y2), (80, 80, 80), -1)
        cv2.rectangle(frame, (x1, y1), (x2, y2), (255,255,255), border)
        
        #cv2.putText(frame, label, (x1+5, y1+38),
                    #cv2.FONT_HERSHEY_SIMPLEX, 0.45, (255,255,255), 2)
        cv2.line(frame,(x1+10, y1+30),(x2-10, y1+30),(255,255,255), thick)  
        buttons.append((x1, y1, x2, y2, 'thickness', thick))

    # CLEAR 버튼 (2행 굵기버튼 옆)
    cx1 = 10 + len(thickness_buttons) * (btn_w + 8) + 10
    cy1, cx2, cy2 = 90, cx1 + btn_w, 150  # y1=90으로 내림
    cv2.rectangle(frame, (cx1, cy1), (cx2, cy2), (0, 0, 180), -1)
    cv2.rectangle(frame, (cx1, cy1), (cx2, cy2), (255,255,255), 1)
    cv2.putText(frame, "CLEAR", (cx1+5, cy1+38),
                cv2.FONT_HERSHEY_SIMPLEX, 0.45, (255,255,255), 2)
    buttons.append((cx1, cy1, cx2, cy2, 'clear', None))

    return buttons


def check_button_click(ix, iy, buttons):
    """검지 좌표가 버튼 영역 안에 있는지 확인"""
    for (x1, y1, x2, y2, btn_type, value) in buttons:
        if x1 < ix < x2 and y1 < iy < y2:
            return btn_type, value
    return None, None


with mp_hands.Hands(
    static_image_mode=False,
    max_num_hands=1,
    model_complexity=1,
    min_detection_confidence=0.5,
    min_tracking_confidence=0.5
) as hands:

    while cap.isOpened():
        success, frame = cap.read()


        ##동영상일때는 break
        if not success:
            break

        #동영상은 주석처리해도 괜찮
        #frame = cv2.flip(frame, 1)
        h, w, _ = frame.shape

        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        frame_rgb.flags.writeable = False
        results = hands.process(frame_rgb)
        frame_rgb.flags.writeable = True
        frame = cv2.cvtColor(frame_rgb, cv2.COLOR_RGB2BGR)

        # UI 패널 그리기 + 버튼 목록 받기
        buttons = draw_panel(frame, w)

        if results.multi_hand_landmarks:
            for hand_landmarks in results.multi_hand_landmarks:

                # 검지 끝 (8번)
                index_tip = hand_landmarks.landmark[8]
                ix = int(index_tip.x * w)
                iy = int(index_tip.y * h)

                # 엄지 끝 (4번)
                thumb_tip = hand_landmarks.landmark[4]
                tx = int(thumb_tip.x * w)
                ty = int(thumb_tip.y * h)

                distance = math.sqrt((ix-tx)**2 + (iy-ty)**2)

                # 검지 커서 표시
                cv2.circle(frame, (ix, iy), 8, current_color, -1)

                if distance < CLICK_DISTANCE:
                    # ── 클릭 상태 ──
                    cv2.circle(frame, (ix, iy), 15, (0, 0, 255), 2)

                    # 패널 영역 클릭인지 확인
                    btn_type, value = check_button_click(ix, iy, buttons)

                    if btn_type == 'color':
                        current_color = value
                        if index_trail:
                            all_trails.append((index_trail.copy(), current_color, current_thickness))
                            index_trail.clear()

                    elif btn_type == 'thickness':
                        current_thickness = value
                        if index_trail:
                            all_trails.append((index_trail.copy(), current_color, current_thickness))
                            index_trail.clear()

                    elif btn_type == 'clear':
                        index_trail.clear()
                        all_trails.clear()

                    else:
                        # 패널 밖 클릭 → 펜 들기
                        if index_trail:
                            all_trails.append((index_trail.copy(), current_color, current_thickness))
                            index_trail.clear()

                else:
                    # ── 그리기 상태 (패널 아래 영역만) ──
                    if iy > PANEL_HEIGHT:
                        index_trail.append((ix, iy))

        else:
            if index_trail:
                all_trails.append((index_trail.copy(), current_color, current_thickness))
                index_trail.clear()

        # 저장된 선 전부 그리기
        for (trail, color, thickness) in all_trails:
            for i in range(1, len(trail)):
                cv2.line(frame, trail[i-1], trail[i], color, thickness)

        # 현재 그리는 선
        for i in range(1, len(index_trail)):
            cv2.line(frame, index_trail[i-1], index_trail[i], current_color, current_thickness)

        # 현재 설정 표시
        cv2.putText(frame, f"Q:종료  C:전체지우기",
                    (10, h-15), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (200,200,200), 1)

        cv2.imshow("electronic board", frame)

        key = cv2.waitKey(5)
        if key == ord('q'):
            break
        elif key == ord('c'):
            index_trail.clear()
            all_trails.clear()

cap.release()
cv2.destroyAllWindows()

In [27]:
import cv2
import mediapipe as mp
import math

mp_drawing = mp.solutions.drawing_utils
mp_drawing_styles = mp.solutions.drawing_styles
mp_hands = mp.solutions.hands

cap = cv2.VideoCapture("hands3.mp4")

CLICK_DISTANCE = 40 #클릭인식거리 
PANEL_HEIGHT   = 160 #상단 패널 높이

current_color     = (0, 255, 0)
current_thickness = 5

index_trail = []
all_trails  = []

color_buttons = [
    ("R", (0, 0, 255),   (0, 0, 255)),
    ("G", (0, 255, 0),   (0, 255, 0)),
    ("B", (255, 0, 0),   (255, 0, 0)),
    ("W", (255,255,255), (200,200,200)),
    ("Y", (0, 255, 255), (0, 255, 255)),
]

thickness_buttons = [
    ("THIN",   1),
    ("NORMAL", 5),
    ("THICK",  10),
]

def draw_panel(frame, w):
    cv2.rectangle(frame, (0, 0), (w, PANEL_HEIGHT), (40, 40, 40), -1)

    btn_w = 70
    buttons = []

    # 색상 버튼 (1행)
    for i, (label, color, btn_color) in enumerate(color_buttons):
        x1 = 10 + i * (btn_w + 8)
        y1, x2, y2 = 10, x1 + btn_w, 70

        border = 3 if current_color == color else 1
        cv2.rectangle(frame, (x1, y1), (x2, y2), btn_color, -1)
        cv2.rectangle(frame, (x1, y1), (x2, y2), (255,255,255), border)
        buttons.append((x1, y1, x2, y2, 'color', color))

    # 굵기 버튼 (2행)
    for i, (label, thick) in enumerate(thickness_buttons):
        x1 = 10 + i * (btn_w + 8)
        y1, x2, y2 = 90, x1 + btn_w, 150

        border = 3 if current_thickness == thick else 1
        cv2.rectangle(frame, (x1, y1), (x2, y2), (80, 80, 80), -1)
        cv2.rectangle(frame, (x1, y1), (x2, y2), (255,255,255), border)
        cv2.line(frame, (x1+10, y1+30), (x2-10, y1+30), (255,255,255), thick)
        buttons.append((x1, y1, x2, y2, 'thickness', thick))

    return buttons


def check_button_click(ix, iy, buttons):
    for (x1, y1, x2, y2, btn_type, value) in buttons:
        if x1 < ix < x2 and y1 < iy < y2:
            return btn_type, value
    return None, None


with mp_hands.Hands(
    static_image_mode=False,
    max_num_hands=1,
    model_complexity=1,
    min_detection_confidence=0.5,
    min_tracking_confidence=0.5
) as hands:

    while cap.isOpened():
        success, frame = cap.read()
        if not success:
            break

        h, w, _ = frame.shape

        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        frame_rgb.flags.writeable = False
        results = hands.process(frame_rgb)
        frame_rgb.flags.writeable = True
        frame = cv2.cvtColor(frame_rgb, cv2.COLOR_RGB2BGR)

        buttons = draw_panel(frame, w)

        if results.multi_hand_landmarks:
            for hand_landmarks in results.multi_hand_landmarks:

                # 검지 끝 (8번)
                ix = int(hand_landmarks.landmark[8].x * w)
                iy = int(hand_landmarks.landmark[8].y * h)

                # 엄지 끝 (4번)
                tx = int(hand_landmarks.landmark[4].x * w)
                ty = int(hand_landmarks.landmark[4].y * h)

                # 중지 끝 (12번)
                mx = int(hand_landmarks.landmark[12].x * w)
                my = int(hand_landmarks.landmark[12].y * h)

                # 검지-엄지 거리 (클릭)
                distance = math.sqrt((ix-tx)**2 + (iy-ty)**2)

                # 검지-중지 거리 (클리어)
                distance_clear = math.sqrt((ix-mx)**2 + (iy-my)**2)

                # 검지 커서
                cv2.circle(frame, (ix, iy), 8, current_color, -1)

                # 검지+중지 만나면 전체 지우기
                if distance_clear < CLICK_DISTANCE:
                    index_trail.clear()
                    all_trails.clear()
                    cv2.putText(frame, "CLEAR!", (ix+10, iy),
                                cv2.FONT_HERSHEY_SIMPLEX, 1, (0,0,255), 2)

                elif distance < CLICK_DISTANCE:
                    # 클릭 상태
                    cv2.circle(frame, (ix, iy), 15, (0, 0, 255), 2)

                    btn_type, value = check_button_click(ix, iy, buttons)

                    if btn_type == 'color':
                        current_color = value
                        if index_trail:
                            all_trails.append((index_trail.copy(), current_color, current_thickness))
                            index_trail.clear()

                    elif btn_type == 'thickness':
                        current_thickness = value
                        if index_trail:
                            all_trails.append((index_trail.copy(), current_color, current_thickness))
                            index_trail.clear()

                else:
                    # 그리기 상태
                    if iy > PANEL_HEIGHT:
                        index_trail.append((ix, iy))

        else:
            if index_trail:
                all_trails.append((index_trail.copy(), current_color, current_thickness))
                index_trail.clear()

        # 저장된 선 전부 그리기
        for (trail, color, thickness) in all_trails:
            for i in range(1, len(trail)):
                cv2.line(frame, trail[i-1], trail[i], color, thickness)

        # 현재 그리는 선
        for i in range(1, len(index_trail)):
            cv2.line(frame, index_trail[i-1], index_trail[i], current_color, current_thickness)

        cv2.imshow("electronic board", frame)

        key = cv2.waitKey(5)
        if key == ord('q'):
            break
        elif key == ord('c'):
            index_trail.clear()
            all_trails.clear()

cap.release()
cv2.destroyAllWindows()